In [1]:
from __future__ import annotations

from pathlib import Path
import shutil
import pandas as pd

RANDOM_STATE = 42
TRAIN_RATIO = 0.8

def first_existing_path(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    pretty = "\n".join(f"- {c.resolve()}" for c in candidates)
    raise FileNotFoundError(f"Could not find any of these paths:\n{pretty}")

# --- Locate input files/folders (works whether cwd is repo root or data_preprocessor/)
real_labels_csv = first_existing_path([
    Path("data/plate_upper_real/labels.csv"),
    Path("../data/plate_upper_real/labels.csv"),
])
real_images_dir = first_existing_path([
    Path("data/plate_upper_real/data_resized_128x32"),
    Path("../data/plate_upper_real/data_resized_128x32"),
    Path("data/plate_upper_real/data"),
    Path("../data/plate_upper_real/data"),
])

synth_labels_csv = first_existing_path([
    Path("data/plate_upper_synth/labels.csv"),
    Path("../data/plate_upper_synth/labels.csv"),
])
synth_images_dir = first_existing_path([
    Path("data/plate_upper_synth/data_resized_128x32"),
    Path("../data/plate_upper_synth/data_resized_128x32"),
    Path("data/plate_upper_synth/data"),
    Path("../data/plate_upper_synth/data"),
])

output_root = first_existing_path([
    Path("data/plate_upper_real"),
    Path("../data/plate_upper_real"),
])
train_dir = output_root / "upper_train"
test_dir = output_root / "upper_test"
train_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

print("real_labels_csv:", real_labels_csv.resolve())
print("real_images_dir:", real_images_dir.resolve())
print("synth_labels_csv:", synth_labels_csv.resolve())
print("synth_images_dir:", synth_images_dir.resolve())
print("train_dir:", train_dir.resolve())
print("test_dir:", test_dir.resolve())

# --- Load labels
df_real = pd.read_csv(real_labels_csv)
if "filename" not in df_real.columns:
    raise ValueError("real labels.csv must contain a 'filename' column")
df_real["filename"] = df_real["filename"].astype(str).str.strip()

# Shuffle once then split 80/20 (train+val/test)
df_real = df_real.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
n_real = len(df_real)
if n_real <= 1:
    n_train = n_real
else:
    n_train = max(1, min(n_real - 1, int(round(n_real * TRAIN_RATIO))))
df_real_train = df_real.iloc[:n_train].copy()
df_real_test = df_real.iloc[n_train:].copy()

# --- Copy helper
def copy_and_filter(df_split: pd.DataFrame, src_dir: Path, dst_dir: Path, label: str) -> pd.DataFrame:
    kept_rows = []
    missing = 0
    for _, row in df_split.iterrows():
        filename = str(row["filename"]).strip()
        src = src_dir / filename
        if not src.exists():
            missing += 1
            continue
        dst = dst_dir / filename
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        kept_rows.append(row)
    out_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    print(f"{label}: kept {len(out_df):,} rows, missing files {missing:,}")
    return out_df

# --- Copy real splits
df_real_train_out = copy_and_filter(df_real_train, real_images_dir, train_dir, "train_real")
df_real_test_out = copy_and_filter(df_real_test, real_images_dir, test_dir, "test_real")

# --- Load synthetic and add to train
df_synth = pd.read_csv(synth_labels_csv)
if "filename" not in df_synth.columns:
    raise ValueError("synthetic labels.csv must contain a 'filename' column")
df_synth["filename"] = df_synth["filename"].astype(str).str.strip()
df_synth_out = copy_and_filter(df_synth, synth_images_dir, train_dir, "train_synth")

# --- Save labels
df_train_full = pd.concat([df_real_train_out, df_synth_out], ignore_index=True)
df_test_full = df_real_test_out

train_labels_path = train_dir / "labels.csv"
test_labels_path = test_dir / "labels.csv"
df_train_full.to_csv(train_labels_path, index=False, encoding="utf-8-sig")
df_test_full.to_csv(test_labels_path, index=False, encoding="utf-8-sig")

print("Saved train labels:", train_labels_path.resolve())
print("Saved test labels:", test_labels_path.resolve())
print("train rows (real+synth):", len(df_train_full))
print("test rows (real only):", len(df_test_full))

real_labels_csv: D:\CodingD\ALPR\data\plate_upper_real\labels.csv
real_images_dir: D:\CodingD\ALPR\data\plate_upper_real\data_resized_128x32
synth_labels_csv: D:\CodingD\ALPR\data\plate_upper_synth\labels.csv
synth_images_dir: D:\CodingD\ALPR\data\plate_upper_synth\data
train_dir: D:\CodingD\ALPR\data\plate_upper_real\upper_train
test_dir: D:\CodingD\ALPR\data\plate_upper_real\upper_test
train_real: kept 8,166 rows, missing files 0
test_real: kept 2,042 rows, missing files 0
train_synth: kept 150,000 rows, missing files 0
Saved train labels: D:\CodingD\ALPR\data\plate_upper_real\upper_train\labels.csv
Saved test labels: D:\CodingD\ALPR\data\plate_upper_real\upper_test\labels.csv
train rows (real+synth): 158166
test rows (real only): 2042
